# TBL_FACT_SALES

In [0]:
from pyspark.sql.functions import *

# ============================================================
# READ SOURCE TABLES
# ============================================================

df_order_details = spark.read.jdbc(
    url=jdbc_url_src,
    table="dbo.ORDER_DETAILS",
    properties=connection_properties_src
)

df_order_header = spark.read.jdbc(
    url=jdbc_url_src,
    table="dbo.ORDER_HEADER",
    properties=connection_properties_src
)

df_return = spark.read.jdbc(
    url=jdbc_url_src,
    table="dbo.RETURN_ITEM",
    properties=connection_properties_src
)

# ============================================================
# READ DIMENSION TABLES
# ============================================================

df_dim_product = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_PRODUCT",
    properties=connection_properties_tgt
)

df_dim_country = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_COUNTRY_LKP",
    properties=connection_properties_tgt
)

df_dim_order_method = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_ORDER_METHOD_LKP",
    properties=connection_properties_tgt
)

df_dim_return_reason = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_RETURN_REASON_LKP",
    properties=connection_properties_tgt
)

df_dim_retailer = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_RETAILER_LKP",
    properties=connection_properties_tgt
)

df_dim_sales_order = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_ORDER",
    properties=connection_properties_tgt
)

df_dim_date = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_DATE",
    properties=connection_properties_tgt
)

# ============================================================
# ACTIVE DIMENSIONS ONLY
# ============================================================

df_dim_product = df_dim_product.filter(col("CHANGE_INFO") == "Y")
df_dim_sales_order = df_dim_sales_order.filter(col("CHANGE_INFO") == "Y")

# ============================================================
# FACT SOURCE JOIN
# ============================================================

df_sales = (
    df_order_details.alias("od")

    .join(
        df_order_header.alias("oh"),
        col("od.ORDER_NUMBER") == col("oh.ORDER_NUMBER"),
        "left"
    )

    .join(
        df_return.alias("rt"),
        col("od.ORDER_DETAIL_CODE") == col("rt.ORDER_DETAIL_CODE"),
        "left"
    )
)

# ============================================================
# JOIN DIMENSIONS
# ============================================================

df_fact_sales = (

    df_sales.alias("src")

    # PRODUCT DIM
    .join(
        df_dim_product.alias("dp"),
        col("src.PRODUCT_NUMBER") == col("dp.PRODUCT_NUMBER"),
        "left"
    )

    # COUNTRY DIM
    .join(
        df_dim_country.alias("dc"),
        col("src.COUNTRY_CODE") == col("dc.COUNTRY_CODE"),
        "left"
    )

    # ORDER METHOD DIM
    .join(
        df_dim_order_method.alias("dom"),
        col("src.ORDER_METHOD_CODE") == col("dom.METHOD_CODE"),
        "left"
    )

    # RETURN REASON DIM
    .join(
        df_dim_return_reason.alias("drr"),
        col("src.RETURN_REASON_CODE") == col("drr.RETURN_REASON_CODE"),
        "left"
    )

    # RETAILER DIM
    .join(
        df_dim_retailer.alias("dr"),
        col("src.RETAILER_SITE_CODE") == col("dr.RETAILER_SITE_CODE"),
        "left"
    )

    # SALES ORDER DIM
    .join(
        df_dim_sales_order.alias("dso"),
        (
            (col("src.ORDER_DETAIL_CODE") == col("dso.ORDER_DETAIL_CODE")) &
            (col("src.ORDER_NUMBER") == col("dso.ORDER_NUMBER"))
        ),
        "left"
    )

    # ORDER DATE DIM
    .join(
        df_dim_date.alias("odd"),
        col("src.ORDER_DATE") == col("odd.FULL_DATE"),
        "left"
    )

    # SHIP DATE DIM
    .join(
        df_dim_date.alias("sdd"),
        col("src.SHIP_DATE") == col("sdd.FULL_DATE"),
        "left"
    )

    # CLOSE DATE DIM
    .join(
        df_dim_date.alias("cdd"),
        col("src.ORDER_CLOSE_DATE") == col("cdd.FULL_DATE"),
        "left"
    )

    .select(

        col("odd.DATE_KEY").alias("ORDER_DATE_KEY"),

        col("dr.RETAILER_KEY"),

        col("dp.PRODUCT_KEY"),

        col("dc.COUNTRY_KEY"),

        col("dom.METHOD_KEY").alias("ORDER_METHOD_KEY"),

        col("dso.ORDER_KEY").alias("SALES_ORDER_KEY"),

        col("sdd.DATE_KEY").alias("SHIP_DATE_KEY"),

        col("drr.RETURN_REASON_KEY"),

        col("cdd.DATE_KEY").alias("CLOSE_DAY_KEY"),

        col("src.RETURNED_QUANTITY"),

        col("src.QUANTITY"),

        col("src.UNIT_COST"),

        col("src.UNIT_PRICE"),

        col("src.UNIT_SALE_PRICE"),

        (
            col("src.QUANTITY") *
            col("src.UNIT_SALE_PRICE")
        ).alias("SALE_TOTAL"),

        lit("INTERNAL DB").alias("SOURCE_ID"),

        current_date().alias("DATA_DATE"),

        current_date().alias("UPDATE_DATE")
    )
)

# ============================================================
# WRITE FACT TABLE
# ============================================================

df_fact_sales.write.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_FACT_SALES",
    mode="append",
    properties=connection_properties_tgt
)

print("FACT SALES LOAD COMPLETED")

# TBL_FACT_INVENTORY

In [0]:
from pyspark.sql.functions import *

# ============================================================
# READ SOURCE TABLE
# ============================================================

df_inventory = spark.read.jdbc(
    url=jdbc_url_src,
    table="dbo.INVENTORY",
    properties=connection_properties_src
)

# ============================================================
# READ DIMENSIONS
# ============================================================

df_dim_product = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_PRODUCT",
    properties=connection_properties_tgt
)

df_dim_warehouse = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_WAREHOUSE_LKP",
    properties=connection_properties_tgt
)

df_dim_year = spark.read.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_DIM_YEAR",
    properties=connection_properties_tgt
)

# ============================================================
# ACTIVE PRODUCT ONLY
# ============================================================

df_dim_product = df_dim_product.filter(
    col("CHANGE_INFO") == "Y"
)

# ============================================================
# FACT INVENTORY
# ============================================================

df_fact_inventory = (

    df_inventory.alias("inv")

    # PRODUCT DIM
    .join(
        df_dim_product.alias("dp"),
        col("inv.PRODUCT_NUMBER") == col("dp.PRODUCT_NUMBER"),
        "left"
    )

    # WAREHOUSE DIM
    .join(
        df_dim_warehouse.alias("dw"),
        col("inv.WAREHOUSE_BRANCH_CODE") == col("dw.WAREHOUSE_BRANCH_CODE"),
        "left"
    )

    # YEAR DIM
    .join(
        df_dim_year.alias("dy"),
        year(col("inv.INVENTORY_DATE")) == col("dy.YEAR_NUMBER"),
        "left"
    )

    .select(

        col("dy.YEAR_KEY").alias("INVENTORY_YEAR_KEY"),

        col("dp.PRODUCT_KEY"),

        col("dw.WAREHOUSE_KEY"),

        col("inv.OPENING_INVENTORY"),

        col("inv.QUANTITY_SALE_SHIP"),

        col("inv.INV_ADDITIONS"),

        col("inv.UNIT_COST"),

        (
            col("inv.OPENING_INVENTORY")
            + col("inv.INV_ADDITIONS")
            - col("inv.QUANTITY_SALE_SHIP")
        ).alias("CLOSING_INVENTORY"),

        lit("INTERNAL DB").alias("SOURCE_ID"),

        current_date().alias("DATA_DATE"),

        current_date().alias("UPDATE_DATE")
    )
)

# ============================================================
# WRITE FACT TABLE
# ============================================================

df_fact_inventory.write.jdbc(
    url=jdbc_url_tgt,
    table="dbo.TBL_FACT_INVENTORY",
    mode="append",
    properties=connection_properties_tgt
)

print("FACT INVENTORY LOAD COMPLETED")
